# Customer Churn Prediction - Phase 1 (Training & Preprocessing)

This notebook handles data loading, preprocessing, and model training. It uses a robust `ColumnTransformer` to ensure data leakage is prevented and the same transformations can be exactly applied during prediction.

In [134]:
import pandas as pd
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
import datetime
import os

In [135]:
os.environ['TENSORBOARD_BINARY'] = r'c:\Users\Admin\Desktop\Coderroots Internship Projects\Customer_Churn-1\venv\Scripts\tensorboard.exe'

## 1. Load Data

In [136]:
df = pd.read_csv('../data/Telco_customer_churn.csv')
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


## 2. Clean and Drop Leakage Columns

In [137]:
drop_columns = [
    'Country', 'State', 'CustomerID', 'Lat Long', 'Latitude', 'Longitude', 
    'Zip Code', 'Churn Score', 'Churn Reason', 'City', 'Churn Label', 'Count'
]
df = df.drop(columns=drop_columns)

# Handle missing values encoded as empty strings in 'Total Charges'
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Gender             7043 non-null   str    
 1   Senior Citizen     7043 non-null   str    
 2   Partner            7043 non-null   str    
 3   Dependents         7043 non-null   str    
 4   Tenure Months      7043 non-null   int64  
 5   Phone Service      7043 non-null   str    
 6   Multiple Lines     7043 non-null   str    
 7   Internet Service   7043 non-null   str    
 8   Online Security    7043 non-null   str    
 9   Online Backup      7043 non-null   str    
 10  Device Protection  7043 non-null   str    
 11  Tech Support       7043 non-null   str    
 12  Streaming TV       7043 non-null   str    
 13  Streaming Movies   7043 non-null   str    
 14  Contract           7043 non-null   str    
 15  Paperless Billing  7043 non-null   str    
 16  Payment Method     7043 non-null   

## 3. Train-Test Split
*Crucial: We perform the split before any imputation or scaling to prevent data leakage.*

In [138]:
X = df.drop('Churn Value', axis=1)
y = df['Churn Value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f'X_train shape: {X_train.shape}, X_test shape: {X_test.shape}')
print(f'\nClass distribution in y_train:\n{y_train.value_counts()}')

X_train shape: (5634, 20), X_test shape: (1409, 20)

Class distribution in y_train:
Churn Value
0    4165
1    1469
Name: count, dtype: int64


## 4. Build Preprocessing Pipeline

In [139]:
binary_features = [
    'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 
    'Streaming TV', 'Streaming Movies', 'Paperless Billing', 'Multiple Lines', 
    'Gender', 'Online Backup', 'Online Security', 'Device Protection', 'Tech Support'
]

nominal_features = ['Internet Service', 'Contract', 'Payment Method']

numerical_features = ['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV']


num_pipeline=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])

bin_pipeline=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1))
])
nom_pipeline=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(handle_unknown='ignore',sparse_output=False))
])

preprocessor=ColumnTransformer(
    transformers=[
        ('num',num_pipeline,numerical_features),
        ('bin',bin_pipeline,binary_features),
        ('nom',nom_pipeline,nominal_features)
    ],
    remainder='drop'
)


## 5. Fit and Transform Data, then Export Preprocessor

In [140]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

os.makedirs('logs', exist_ok=True)
with open('preprocessor.pkl', 'wb') as file:
    pickle.dump(preprocessor, file)

print(f'Processed X_train shape: {X_train_processed.shape}')

Processed X_train shape: (5634, 27)


## 6. Model Training

In [141]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_processed.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

opt = tf.keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

c:\Users\Admin\Desktop\Coderroots Internship Projects\Customer_Churn-1\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [142]:
log_dir = 'logs/fit/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train_processed, y_train, 
    validation_data=(X_test_processed, y_test), 
    epochs=100,
    batch_size=32,
    callbacks=[tensorboard_callback, early_stopping_callback],
    verbose=1
)

model.save('model.h5')

Epoch 1/100
177/177 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.7991 - auc: 0.8347 - loss: 0.4247 - val_accuracy: 0.7999 - val_auc: 0.8521 - val_loss: 0.4206
Epoch 2/100
177/177 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8078 - auc: 0.8527 - loss: 0.4052 - val_accuracy: 0.8013 - val_auc: 0.8533 - val_loss: 0.4344
Epoch 3/100
177/177 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8108 - auc: 0.8572 - loss: 0.3998 - val_accuracy: 0.7991 - val_auc: 0.8570 - val_loss: 0.4168
Epoch 4/100
177/177 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8140 - auc: 0.8596 - loss: 0.3962 - val_accuracy: 0.8006 - val_auc: 0.8568 - val_loss: 0.4182
Epoch 5/100
177/177 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8115 - auc: 0.8610 - loss: 0.3956 - val_accuracy: 0.7991 - val_auc: 0.8605 - val_loss: 0.4147
Epoch 6/100
177/177 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8136 - auc: 0.8652 - loss: 0.3894 - val_accuracy: 0.8013 - val_auc: 0.8527 - val_loss: 0.4299
Epoch 7/100
177/177 ━━━━━━━━━━━━━━

## 7. Model Evaluation

In [143]:
y_pred_prob = model.predict(X_test_processed)
y_pred = (y_pred_prob >= 0.5).astype(int)

print('Test Accuracy:', accuracy_score(y_test, y_pred))
print('ROC-AUC Score:', roc_auc_score(y_test, y_pred_prob))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred))

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Test Accuracy: 0.7991483321504613
ROC-AUC Score: 0.860406342913776

Confusion Matrix:
 [[943  66]
 [217 183]]

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.93      0.87      1009
           1       0.73      0.46      0.56       400

    accuracy                           0.80      1409
   macro avg       0.77      0.70      0.72      1409
weighted avg       0.79      0.80      0.78      1409



In [144]:
df

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,...,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value,CLTV
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,3239
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,2701
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,...,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1,5372
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,...,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1,5003
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,...,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1,5340
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Female,No,No,No,72,Yes,No,No,No internet service,No internet service,...,No internet service,No internet service,No internet service,Two year,Yes,Bank transfer (automatic),21.15,1419.40,0,5306
7039,Male,No,Yes,Yes,24,Yes,Yes,DSL,Yes,No,...,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,0,2140
7040,Female,No,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,...,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,0,5560
7041,Female,No,Yes,Yes,11,No,No phone service,DSL,Yes,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,0,2793
